# Task 3 - Neural Networks

Dans ce notebook, nous allons implémenter un réseau de neurones artificiels avec TensorFlow/Keras, l'un des modèles les plus puissants en machine learning, inspirés du fonctionnement du cerveau humain.

## 1. Importation des bibliothèques nécessaires

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import warnings
warnings.filterwarnings('ignore')

# Configuration de l'affichage
%matplotlib inline

# Pour la reproductibilité
np.random.seed(42)
tf.random.set_seed(42)

## 2. Chargement et exploration des données

Pour cette démonstration, nous utiliserons le jeu de données du cancer du sein de Wisconsin, qui est un problème de classification binaire classique.

In [ ]:
# Chargement des données du cancer du sein de Wisconsin
breast_cancer = load_breast_cancer()

# Création d'un DataFrame pandas
df = pd.DataFrame(data=breast_cancer.data, columns=breast_cancer.feature_names)
df['target'] = breast_cancer.target

df.head()

In [ ]:
# Exploration des données
print("Shape of dataset:", df.shape)
print("\nInfo of dataset:")
df.info()

In [ ]:
# Statistiques descriptives
df.describe()

In [ ]:
# Vérification de la distribution des classes (variable cible)
target_column = df.columns[-1]
print(f"Variable cible: {target_column}")
print("\nDistribution des classes:")
print(df[target_column].value_counts())

# Visualisation de la distribution des classes
plt.figure(figsize=(8, 6))
df[target_column].value_counts().plot(kind='bar')
plt.title('Distribution des classes')
plt.xlabel('Classes')
plt.ylabel('Nombre d\'échantillons')
plt.xticks(rotation=0)
plt.grid(True, alpha=0.3)
plt.show()

## 3. Prétraitement des données

Séparation des caractéristiques (features) et de la variable cible, puis normalisation des données.

In [ ]:
# Séparation des caractéristiques (features) et de la variable cible
X = df.iloc[:, :-1]  # Toutes les colonnes sauf la dernière
y = df.iloc[:, -1]   # Dernière colonne (variable cible)

print(f"Dimensions des caractéristiques: {X.shape}")
print(f"Dimensions de la variable cible: {y.shape}")

In [ ]:
# Normalisation des données (recommandée pour les réseaux de neurones)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convertir en DataFrame pour faciliter la manipulation
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
print("Données après normalisation:")
X_scaled.head()

## 4. Division des données

Division des données en ensembles d'entraînement, de validation et de test.

In [ ]:
# Division des données en ensembles d'entraînement et test
X_temp, X_test, y_temp, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

# Division de l'ensemble temporaire en ensembles d'entraînement et de validation
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42, stratify=y_temp)

print(f"Taille de l'ensemble d'entraînement: {X_train.shape}")
print(f"Taille de l'ensemble de validation: {X_val.shape}")
print(f"Taille de l'ensemble de test: {X_test.shape}")

## 5. Construction du modèle de réseau de neurones

Création d'une architecture de réseau de neurones avec TensorFlow/Keras.

In [ ]:
# Construction du modèle
model = keras.Sequential([
    # Couche d'entrée
    layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dropout(0.3),  # Dropout pour la régularisation
    
    # Première couche cachée
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),
    
    # Deuxième couche cachée
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.2),
    
    # Couche de sortie
    layers.Dense(1, activation='sigmoid')  # Sigmoid pour classification binaire
])

# Compilation du modèle
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Affichage de la structure du modèle
model.summary()

## 6. Entraînement du modèle

Entraînement du réseau de neurones avec suivi de la progression.

In [ ]:
# Définition des callbacks
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=5,
    min_lr=0.001
)

In [ ]:
# Entraînement du modèle
history = model.fit(
    X_train, y_train,
    batch_size=32,
    epochs=100,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

## 7. Visualisation des courbes d'apprentissage

Analyse de l'évolution de la perte et de la précision pendant l'entraînement.

In [ ]:
# Visualisation de la perte
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Perte d\'entraînement')
plt.plot(history.history['val_loss'], label='Perte de validation')
plt.title('Perte pendant l\'entraînement')
plt.xlabel('Epoch')
plt.ylabel('Perte')
plt.legend()
plt.grid(True, alpha=0.3)

# Visualisation de la précision
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Précision d\'entraînement')
plt.plot(history.history['val_accuracy'], label='Précision de validation')
plt.title('Précision pendant l\'entraînement')
plt.xlabel('Epoch')
plt.ylabel('Précision')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Évaluation du modèle

Évaluation des performances du modèle sur l'ensemble de test.

In [ ]:
# Évaluation sur l'ensemble de test
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Perte sur l'ensemble de test: {test_loss:.4f}")
print(f"Précision sur l'ensemble de test: {test_accuracy:.4f}")

In [ ]:
# Prédiction sur l'ensemble de test
y_pred_proba = model.predict(X_test)
y_pred = (y_pred_proba > 0.5).astype(int).flatten()

# Affichage des premières prédictions
results_df = pd.DataFrame({'Valeurs réelles': y_test, 'Prédictions': y_pred, 'Probabilités': y_pred_proba.flatten()})
print("Comparaison des valeurs réelles et prédites:")
print(results_df.head(10))

In [ ]:
# Rapport de classification détaillé
print("\nRapport de classification:")
print(classification_report(y_test, y_pred))

In [ ]:
# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Matrice de confusion')
plt.xlabel('Prédictions')
plt.ylabel('Valeurs réelles')
plt.show()

In [ ]:
# Courbe ROC et AUC
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Courbe ROC (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Classifieur aléatoire')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Taux de faux positifs')
plt.ylabel('Taux de vrais positifs')
plt.title('Courbe ROC')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

print(f"Aire sous la courbe ROC (AUC): {roc_auc:.4f}")

## 9. Identification et résolution du sur-apprentissage

Analyse des signes de sur-apprentissage et techniques pour l'atténuer.

In [ ]:
# Analyse de l'écart entre les performances d'entraînement et de validation
train_acc = history.history['accuracy'][-1]
val_acc = history.history['val_accuracy'][-1]
train_loss = history.history['loss'][-1]
val_loss = history.history['val_loss'][-1]

print(f"Précision d'entraînement finale: {train_acc:.4f}")
print(f"Précision de validation finale: {val_acc:.4f}")
print(f"Écart de précision: {abs(train_acc - val_acc):.4f}")

print(f"\nPerte d'entraînement finale: {train_loss:.4f}")
print(f"Perte de validation finale: {val_loss:.4f}")
print(f"Écart de perte: {abs(train_loss - val_loss):.4f}")

# Si l'écart est significatif, il peut y avoir du sur-apprentissage
if abs(train_acc - val_acc) > 0.05:
    print("\n⚠️  Signes possibles de sur-apprentissage détectés")
else:
    print("\n✅ Pas de sur-apprentissage significatif détecté")

## 10. Amélioration du modèle

Création d'une version améliorée du modèle avec des techniques supplémentaires de régularisation.

In [ ]:
# Modèle amélioré avec plus de régularisation
improved_model = keras.Sequential([
    # Couche d'entrée avec BatchNormalization
    layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    
    # Première couche cachée
    layers.Dense(32, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    
    # Deuxième couche cachée
    layers.Dense(16, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    
    # Couche de sortie
    layers.Dense(1, activation='sigmoid')
])

# Compilation du modèle amélioré
improved_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Structure du modèle amélioré:")
improved_model.summary()

In [ ]:
# Entraînement du modèle amélioré
improved_history = improved_model.fit(
    X_train, y_train,
    batch_size=32,
    epochs=100,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

In [ ]:
# Évaluation du modèle amélioré
improved_test_loss, improved_test_accuracy = improved_model.evaluate(X_test, y_test, verbose=0)
print(f"\nPerformances du modèle amélioré sur l'ensemble de test:")
print(f"Perte: {improved_test_loss:.4f}")
print(f"Précision: {improved_test_accuracy:.4f}")

print(f"\nComparaison avec le modèle initial:")
print(f"Amélioration de la précision: {improved_test_accuracy - test_accuracy:.4f}")

## 11. Visualisation des résultats du modèle amélioré

In [ ]:
# Visualisation des courbes d'apprentissage du modèle amélioré
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(improved_history.history['loss'], label='Perte d\'entraînement')
plt.plot(improved_history.history['val_loss'], label='Perte de validation')
plt.title('Perte pendant l\'entraînement (Modèle amélioré)')
plt.xlabel('Epoch')
plt.ylabel('Perte')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(improved_history.history['accuracy'], label='Précision d\'entraînement')
plt.plot(improved_history.history['val_accuracy'], label='Précision de validation')
plt.title('Précision pendant l\'entraînement (Modèle amélioré)')
plt.xlabel('Epoch')
plt.ylabel('Précision')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Conclusion

Dans ce notebook, nous avons implémenté un réseau de neurones artificiels avec TensorFlow/Keras :
- Chargement et exploration des données du cancer du sein
- Prétraitement des données avec normalisation
- Division des données en ensembles d'entraînement, de validation et de test
- Construction d'un modèle de réseau de neurones avec plusieurs couches
- Entraînement du modèle avec suivi des courbes d'apprentissage
- Évaluation des performances avec plusieurs métriques (précision, rappel, F1-score, AUC)
- Visualisation des résultats (matrice de confusion, courbe ROC)
- Identification et atténuation du sur-apprentissage
- Amélioration du modèle avec des techniques avancées de régularisation

Les réseaux de neurones sont des modèles très puissants capables d'apprendre des représentations complexes dans les données. Cependant, ils nécessitent une attention particulière pour éviter le sur-apprentissage, notamment grâce à des techniques comme le dropout, la normalisation par lots et l'arrêt anticipé. La validation croisée et l'ensemble de validation sont essentiels pour surveiller les performances et ajuster les hyperparamètres.